In [ ]:
# 导入必要的库
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from scipy.optimize import minimize

In [ ]:
# 加载数据集A（TRAIN38.mat的data）
def load_dataset_A(file_path):
    """加载TRAIN38.mat中的data作为数据集A"""
    print(f"加载数据集A: {file_path}")
    
    try:
        f = h5py.File(file_path, 'r')
        # 从文件中读取data矩阵
        data = np.array(f['data'])
        
        # 读取prob_idx以便后续匹配
        prob_idx = np.array(f['prob_idx'])
        
        # 转置data（根据原代码的处理方式）
        data = data.transpose()
        prob_idx = prob_idx.transpose()
        
        f.close()
        
        print(f"数据集A加载完成:")
        print(f"  数据形状: {data.shape}")
        print(f"  prob_idx形状: {prob_idx.shape}")
        print(f"  数据类型: {data.dtype}")
        print(f"  统计信息: min={np.min(data)}, max={np.max(data)}, mean={np.mean(data)}, std={np.std(data)}")
        
        return {
            'data': data,
            'prob_idx': prob_idx
        }
    
    except Exception as e:
        print(f"加载数据集A时出错: {e}")
        return None

# 加载数据集B（merged目录下的数据）
def load_dataset_B(base_dir):
    """加载merged目录下的所有npy文件作为数据集B"""
    print(f"加载数据集B: {base_dir}")
    
    merged_dir = os.path.join(base_dir, 'merged')
    
    # 检查目录是否存在
    if not os.path.exists(merged_dir):
        print(f"目录不存在: {merged_dir}")
        return None
    
    # 获取所有标签文件
    voxel_files = glob.glob(os.path.join(merged_dir, "label_*_count_*_voxels.npy"))
    
    if not voxel_files:
        print(f"在 {merged_dir} 中未找到体素文件")
        return None
    
    # 加载所有体素数据并合并
    all_data = []
    all_labels = []
    
    for voxel_file in tqdm(voxel_files, desc=f"加载merged数据"):
        # 从文件名提取标签ID
        filename = os.path.basename(voxel_file)
        parts = filename.split('_')
        label_id = int(parts[1])
        
        # 加载体素数据
        voxels = np.load(voxel_file)
        
        # 将标签ID扩展为与体素数据相同的行数
        labels = np.full((voxels.shape[0], 1), label_id)
        
        all_data.append(voxels)
        all_labels.append(labels)
    
    if all_data:
        combined_data = np.vstack(all_data)
        combined_labels = np.vstack(all_labels)
        
        print(f"数据集B加载完成:")
        print(f"  数据形状: {combined_data.shape}")
        print(f"  标签形状: {combined_labels.shape}")
        print(f"  数据类型: {combined_data.dtype}")
        print(f"  统计信息: min={np.min(combined_data)}, max={np.max(combined_data)}, mean={np.mean(combined_data)}, std={np.std(combined_data)}")
        
        return {
            'data': combined_data,
            'labels': combined_labels
        }
    else:
        print("未加载到任何数据")
        return None

# 加载数据
file_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'
restructured_base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured'

dataset_A_result = None
dataset_B_result = None

if os.path.exists(file_path):
    dataset_A_result = load_dataset_A(file_path)
else:
    print(f"文件不存在: {file_path}")

if os.path.exists(restructured_base_dir):
    dataset_B_result = load_dataset_B(restructured_base_dir)
else:
    print(f"目录不存在: {restructured_base_dir}")

In [ ]:
# 优化的StandardScaler实现 - 使用分布匹配而不是直接MSE
class DistributionScaler:
    """优化的StandardScaler，调整mean和std参数以使两个数据集的分布尽可能相似"""
    
    def __init__(self, initial_mean=None, initial_std=None):
        """初始化DistributionScaler"""
        self.mean_ = initial_mean
        self.scale_ = initial_std
        self.n_features_ = None
    
    def transform(self, X):
        """使用当前参数转换X"""
        X = np.asarray(X)
        return (X - self.mean_) / self.scale_
    
    def fit_transform(self, X, **kwargs):
        """先fit再transform"""
        return self.transform(X)
    
    def distribution_distance(self, X, target_X, n_samples=10000):
        """
        计算两个数据集分布之间的距离
        使用随机抽样和KL散度/Wasserstein距离的近似
        """
        # 随机抽样
        X_idx = np.random.choice(X.shape[0], min(n_samples, X.shape[0]), replace=False)
        target_idx = np.random.choice(target_X.shape[0], min(n_samples, target_X.shape[0]), replace=False)
        
        X_sample = X[X_idx]
        target_sample = target_X[target_idx]
        
        # 转换X
        X_transformed = self.transform(X_sample)
        
        # 计算分布距离 - 这里使用简化的均值和方差距离
        # 可以替换为更复杂的分布距离度量
        mean_dist = np.mean((np.mean(X_transformed, axis=0) - np.mean(target_sample, axis=0))**2)
        std_dist = np.mean((np.std(X_transformed, axis=0) - np.std(target_sample, axis=0))**2)
        
        # 总距离是均值距离和标准差距离的加权和
        total_dist = mean_dist + std_dist
        
        return total_dist
    
    def fit(self, X, target_X, method='L-BFGS-B', max_iter=1000, verbose=True, n_samples=10000):
        """
        使用SciPy的优化器优化参数，使两个数据集的分布尽可能相似
        
        参数:
            X: 原始数据集
            target_X: 目标数据集
            method: SciPy优化方法
            max_iter: 最大迭代次数
            verbose: 是否打印进度信息
            n_samples: 用于计算分布距离的样本数
        """
        # 检查输入
        X = np.asarray(X)
        target_X = np.asarray(target_X)
        
        if X.shape[1] != target_X.shape[1]:
            raise ValueError(f"X与target_X特征数不同: {X.shape[1]} vs {target_X.shape[1]}")
        
        n_features = X.shape[1]
        self.n_features_ = n_features
        
        # 初始化参数
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
        
        if self.scale_ is None:
            self.scale_ = np.std(X, axis=0)
            # 防止除以零
            self.scale_[self.scale_ == 0] = 1.0
        
        # 准备记录迭代过程
        self.loss_history_ = []
        
        # 定义目标函数
        def objective(params):
            mean_params = params[:n_features]
            std_params = params[n_features:]
            
            # 确保std为正
            std_params = np.maximum(std_params, 1e-10)
            
            # 临时设置参数
            self.mean_ = mean_params
            self.scale_ = std_params
            
            # 计算分布距离
            dist = self.distribution_distance(X, target_X, n_samples)
            self.loss_history_.append(dist)
            
            return dist
        
        # 优化
        initial_params = np.concatenate([self.mean_, self.scale_])
        
        if verbose:
            print("开始SciPy优化...")
            print(f"初始分布距离: {self.distribution_distance(X, target_X, n_samples):.6f}")
        
        result = minimize(
            objective, 
            initial_params, 
            method=method,
            options={'disp': verbose, 'maxiter': max_iter}
        )
        
        if verbose:
            print(f"优化完成: {result.message}")
            print(f"最终分布距离: {result.fun:.6f}")
            print(f"迭代次数: {len(self.loss_history_)}")
        
        # 更新参数
        self.mean_ = result.x[:n_features]
        self.scale_ = np.maximum(result.x[n_features:], 1e-10)
        
        return self
    
    def save(self, file_path):
        """保存优化后的scaler到文件"""
        if not hasattr(self, 'mean_') or not hasattr(self, 'scale_'):
            print("无法保存scaler，请先进行fit")
            return False
        
        # 准备要保存的数据
        data = {
            'mean_': self.mean_,
            'scale_': self.scale_,
            'n_features_': self.n_features_
        }
        
        if hasattr(self, 'loss_history_'):
            data['loss_history_'] = self.loss_history_
        
        try:
            np.savez(file_path, **data)
            print(f"优化后的scaler已保存到: {file_path}")
            return True
        except Exception as e:
            print(f"保存scaler时出错: {e}")
            return False
    
    @classmethod
    def load(cls, file_path):
        """从文件加载优化后的scaler"""
        try:
            data = np.load(file_path)
            scaler = cls()
            scaler.mean_ = data['mean_']
            scaler.scale_ = data['scale_']
            scaler.n_features_ = data['n_features_']
            
            if 'loss_history_' in data:
                scaler.loss_history_ = data['loss_history_']
            
            print(f"优化后的scaler已从 {file_path} 加载")
            return scaler
        except Exception as e:
            print(f"加载scaler时出错: {e}")
            return None

In [ ]:
# 应用优化的scaler
def apply_distribution_scaler(dataset_A_result, dataset_B_result):
    """应用DistributionScaler将数据集A转换为与数据集B分布相似"""
    if dataset_A_result is None or dataset_B_result is None:
        print("无法应用scaler，请确保两个数据集已正确加载")
        return
    
    A = dataset_A_result['data']
    B = dataset_B_result['data']
    
    print("\n=== 应用DistributionScaler ===")
    
    # 检查特征数量是否相同
    if A.shape[1] == B.shape[1]:
        # 使用标准的StandardScaler作为基准
        print("\n使用标准StandardScaler:")
        standard_scaler = StandardScaler()
        standard_scaler.fit(A)
        
        # 创建优化的DistributionScaler
        print("\n使用优化的DistributionScaler:")
        dist_scaler = DistributionScaler()
        
        # 计算拟合前的分布距离
        init_dist = dist_scaler.distribution_distance(A, B)
        print(f"初始分布距离: {init_dist:.6f}")
        
        # 拟合scaler
        dist_scaler.fit(A, B, verbose=True, n_samples=10000, max_iter=50)
        
        # 计算拟合后的分布距离
        final_dist = dist_scaler.distribution_distance(A, B)
        print(f"最终分布距离: {final_dist:.6f}")
        print(f"相对初始距离的改进: {(1 - final_dist/init_dist) * 100:.2f}%")
        
        # 可视化比较结果
        print("\n可视化比较:")
        
        # 绘制损失曲线
        plt.figure(figsize=(10, 6))
        plt.plot(dist_scaler.loss_history_, label='优化过程中的分布距离')
        plt.axhline(y=init_dist, color='r', linestyle='-', label='初始分布距离')
        plt.xlabel('迭代次数')
        plt.ylabel('分布距离')
        plt.title('优化过程中的分布距离变化')
        plt.legend()
        plt.grid(True)
        plt.show()
        
        # 比较分布（随机抽样）
        vis_sample_size = 10000
        A_idx = np.random.choice(A.shape[0], min(vis_sample_size, A.shape[0]), replace=False)
        B_idx = np.random.choice(B.shape[0], min(vis_sample_size, B.shape[0]), replace=False)
        
        A_sample = A[A_idx]
        B_sample = B[B_idx]
        
        # 使用标准scaler转换
        A_std = standard_scaler.transform(A_sample)
        
        # 使用优化的scaler转换
        A_opt = dist_scaler.transform(A_sample)
        
        # 绘制特征分布比较
        n_features_to_plot = min(5, A.shape[1])
        fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(12, 4*n_features_to_plot))
        
        for i in range(n_features_to_plot):
            feature_idx = i
            
            if n_features_to_plot == 1:
                ax = axes
            else:
                ax = axes[i]
            
            ax.hist(A_sample[:, feature_idx], bins=50, alpha=0.3, label='原始数据集A')
            ax.hist(A_std[:, feature_idx], bins=50, alpha=0.3, label='标准Scaler')
            ax.hist(A_opt[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler')
            ax.hist(B_sample[:, feature_idx], bins=50, alpha=0.3, label='目标数据集B')
            ax.set_title(f'特征 {feature_idx+1} 分布比较')
            ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        # 保存优化后的scaler
        dist_scaler.save('distribution_scaler.npz')
        
        return dist_scaler
    else:
        print(f"警告: 两个数据集的特征数量不同! A: {A.shape[1]}, B: {B.shape[1]}")
        return None

# 如果已加载数据集A和数据集B，则应用优化的scaler
if dataset_A_result is not None and dataset_B_result is not None:
    best_scaler = apply_distribution_scaler(dataset_A_result, dataset_B_result)

In [ ]:
# 使用优化的scaler示例
def scaler_usage_example(best_scaler=None):
    """展示如何使用优化的scaler进行数据转换"""
    print("\n=== 优化的Scaler使用示例 ===")
    
    # 如果没有提供scaler，则尝试加载
    if best_scaler is None:
        if os.path.exists('distribution_scaler.npz'):
            best_scaler = DistributionScaler.load('distribution_scaler.npz')
        else:
            print("未找到保存的scaler文件，请先运行优化")
            return
    
    # 生成一些测试数据
    np.random.seed(42)
    test_data = np.random.randn(1000, best_scaler.n_features_)
    
    # 使用优化的scaler转换数据
    transformed_data = best_scaler.transform(test_data)
    
    # 显示转换前后的统计信息
    print("\n转换前的统计信息:")
    print(f"  均值: {np.mean(test_data):.4f}")
    print(f"  标准差: {np.std(test_data):.4f}")
    
    print("\n转换后的统计信息:")
    print(f"  均值: {np.mean(transformed_data):.4f}")
    print(f"  标准差: {np.std(transformed_data):.4f}")
    
    # 可视化转换前后的分布（随机选择几个特征）
    n_features_to_plot = min(3, best_scaler.n_features_)
    feature_indices = np.random.choice(best_scaler.n_features_, n_features_to_plot, replace=False)
    
    fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(10, 4*n_features_to_plot))
    
    for i, feature_idx in enumerate(feature_indices):
        if n_features_to_plot == 1:
            ax = axes
        else:
            ax = axes[i]
        
        ax.hist(test_data[:, feature_idx], bins=30, alpha=0.5, label='原始数据')
        ax.hist(transformed_data[:, feature_idx], bins=30, alpha=0.5, label='转换后数据')
        ax.set_title(f'特征 {feature_idx+1} 转换前后分布比较')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\n实际应用中的代码:")
    print("""
    # 1. 加载保存的scaler
    optimized_scaler = DistributionScaler.load('distribution_scaler.npz')
    
    # 2. 使用scaler转换新数据
    transformed_data = optimized_scaler.transform(new_data)
    
    # 3. 后续处理...
    """)

# 如果已获取最佳scaler，则运行使用示例
if 'best_scaler' in globals() and best_scaler is not None:
    scaler_usage_example(best_scaler)